Notebook để xử lý nhiều dạng tài liệu nội bộ của OU:
- Số tay sinh viên / tài liệu hướng dẫn: chunk theo phần, mục, tiểu mục
- Quy chế đào tạo: chunk theo Chương - Điều - Khoản
- Chương trình đào tạo: chunk theo thông tin ngành, mục tiêu, chuẩn đầu ra, môn học
- Học phí / kế hoạch học vụ: chunk theo bảng, sự kiện, mốc thời gian

=> Tạo JSONL co metadata rõ ràng cho chatbot RAG truy xuất chính xác hơn, hạn chế lỗi do cắt ngang câu / cắt ngang bảng.
- Mỗi chunk có thêm trường metadata dạng dictionary để đưa thẳng vào FAISS/Chroma/LangChain
- Metadata có thêm thông tin ngữ nghĩa: mã môn học, tên học, tín chỉ, điều khoản, chương, mốc thời guan, học phí, ngành, mã ngành nếu trích được
- Có kiểm tra chất lượng metadata trước khi đưa vào vector database.

# **0. Cài đặt thư viện**

Cài đặt các thư viện cần thiết để đọc PDF, OCR file scan và xuất JSONL
- `docling`: đọc PDF và chuyển thành Markdown/structured document.
- `pymupdf`: chỉ dùng phụ trợ để đếm số trang nếu cần, không dùng để trích text chính.
- `tqdm`: hiển thị tiến trình xử lý nhiều file.

In [ ]:
# Cài thư viện cần thiết cho pipeline Docling
# Nếu bạn đã cài rồi, cell này có thể bỏ qua.
!pip install -U docling pymupdf tqdm

In [ ]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
print("Docling OK")

# **1. Import thư viện và cấu hình thư mục**

=> KHai báo thư viện, cấu trúc thư mục input/output
- Upload file vào thư mục pdf_inputs
- Kết quả nằm trong rag_outputs
- Pipeline mới dùng Docling để chuyển PDF thành Markdown, sau đó mới chunk.
- Cấu hình kích thước chunk mặc định và overlap

In [ ]:


import re
import json
import shutil
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from tqdm.auto import tqdm

# Docling: thư viện chính để chuyển PDF sang Markdown/structured document
from docling.document_converter import DocumentConverter

# PyMuPDF chỉ dùng phụ trợ, ví dụ đếm số trang PDF.
# Không dùng PyMuPDF làm nguồn text chính nữa.
try:
    import fitz
except Exception:
    fitz = None

# =========================
# CẤU HÌNH THƯ MỤC
# =========================
INPUT_DIR = Path("pdf_inputs")
OUTPUT_DIR = Path("rag_outputs")
TEXT_DIR = OUTPUT_DIR / "texts"
JSONL_DIR = OUTPUT_DIR / "jsonl"
REPORT_DIR = OUTPUT_DIR / "reports"
MARKDOWN_DIR = OUTPUT_DIR / "markdown"

for folder in [INPUT_DIR, TEXT_DIR, JSONL_DIR, REPORT_DIR, MARKDOWN_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# =========================
# CẤU HÌNH CHUNK
# =========================
# Kích thước chunk fallback khi không nhận diện được cấu trúc.
DEFAULT_CHUNK_SIZE = 1200
DEFAULT_OVERLAP = 180

# Khi Docling xuất Markdown quá dài, ta có thể chia thành các đoạn logic lớn trước khi đưa vào các hàm chunk chuyên biệt.
DOC_LOGICAL_PAGE_SIZE = 8000

# Giữ lại biến này để các hàm cũ không bị lỗi tham số.
# Trong pipeline Docling, zoom không còn ảnh hưởng trực tiếp như EasyOCR.
OCR_ZOOM = 3

print("Input folder:", INPUT_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())



# **2. Khởi tạo Docling**

=> Khởi tạo DocumentConverter của Docling.


Điểm mạnh của Docling là giữ heading, bảng và layout tốt hơn OCR thuần, phù hợp hơn cho RAG.

In [ ]:
# Khởi tạo Docling converter.
# Cell này thường chạy nhanh. Việc convert PDF sẽ diễn ra ở cell xử lý chính.
converter = DocumentConverter()
print("Docling converter đã sẵn sàng")


# **3. Hàm tiện ích: chuẩn hóa tên file, text + metadata**

=> Các hàm chung cho toàn bộ pipline:
- Chuẩn hóa tên file để tạo output an toàn
- Chuẩn hóa khoảng trắng trong văn bản
- Xóa artifact Markdown không cần thiết
- Sửa một số lỗi OCR tiếng Việt phổ biến
- Tạo cấu trúc chunk chuẩn bằng hàm make_chunk.

Metadata gồm 2 lớp:
- Các field top-level: source, page_start, document_type, chunk_type, title để dễ đọc trực tiếp
- Trường metadata chứa toàn bộ thông tin nguồn + thông tin ngữ nghĩa để đưa vào vector database.

VD: 1 chunkk 1 môn sẽ có metadata
```json
{
  "source_file": "ctdt_CongNgheThongTin.PDF",
  "page_start": 6,
  "document_type": "curriculum",
  "chunk_type": "course",
  "course_code": "ITEC2502",
  "course_name": "Cơ sở dữ liệu",
  "credits": 3,
  "major_vi": "Công Nghệ Thông Tin",
  "major_code": "7480201"
}

In [ ]:
def clean_filename(name: str) -> str:
    """Chuẩn hóa tên file để tạo output an toàn.

    Ví dụ: "Học phí 2026.pdf" -> "Học_phí_2026"
    """
    stem = Path(name).stem
    stem = re.sub(r"[^a-zA-Z0-9_\-À-ỹ]", "_", stem)
    stem = re.sub(r"_+", "_", stem).strip("_")
    return stem or "document"



def normalize_spaces(text: str) -> str:
    """Chuẩn hóa khoảng trắng nhưng vẫn giữ xuống dòng để nhận diện cấu trúc."""
    text = text.replace("\x0c", " ")
    text = text.replace("\ufeff", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def remove_markdown_noise(text: str) -> str:
  """Xóa các artifact không có ích cho RAG sau khi Docling xuất Markdown."""
  if not text:
        return ""

  text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)
  text = re.sub(r"!\[.*?\]\(.*?\)", " ", text)
  text = re.sub(r"\[IMAGE\]", " ", text, flags=re.IGNORECASE)
  text = re.sub(r"^\s*[-_\\]{1,3}\s*$", " ", text, flags=re.MULTILINE)
  return text

VIETNAMESE_OCR_REPLACEMENTS = {
    "BO GIAO DUC": "BỘ GIÁO DỤC",
    "BO GIÁO DỤC": "BỘ GIÁO DỤC",
    "VA DA0 TAO": "VÀ ĐÀO TẠO",
    "VA DAO TAO": "VÀ ĐÀO TẠO",
    "DA0 TAO": "ĐÀO TẠO",
    "DAO TAO": "ĐÀO TẠO",
    "TWANG DAI HOC MO": "TRƯỜNG ĐẠI HỌC MỞ",
    "TRUONG DAI HOC MO": "TRƯỜNG ĐẠI HỌC MỞ",
    "DAI HOC MO": "ĐẠI HỌC MỞ",
    "THANH PHO HO CHI MINH": "THÀNH PHỐ HỒ CHÍ MINH",
    "HO CHI MINH": "HỒ CHÍ MINH",
    "CONG HOA XA HOI CHU NGHIA": "CỘNG HÒA XÃ HỘI CHỦ NGHĨA",
    "Doc lap": "Độc lập",
    "Tu do": "Tự do",
    "Hanh phuc": "Hạnh phúc",
    "Hanh phtic": "Hạnh phúc",
    "Tự do -114nh phtic": "Tự do - Hạnh phúc",

    "KE HOACH": "KẾ HOẠCH",
    "Ke hoach": "Kế hoạch",
    "ke hoach": "kế hoạch",
    "dao tao": "đào tạo",
    "dai hoc": "đại học",
    "d4i hoc": "đại học",
    "chinh quy": "chính quy",
    "hinh thuc": "hình thức",
    "hinh thtic": "hình thức",
    "trinh do": "trình độ",
    "hoc ky": "học kỳ",
    "học kSi": "học kỳ",
    "hoc IcS7": "học kỳ",
    "Hoc ick": "Học kỳ",
    "FOC 14": "Học kỳ",
    "IOC 143": "Học kỳ 3",
    "H9C 14": "Học kỳ",
    "IIQC 14": "Học kỳ",
    "119c 14": "Học kỳ",
    "HQC14": "Học kỳ",
    "HQCkj7": "Học kỳ",
    "thai gian": "thời gian",
    "Thai gian": "Thời gian",
    "thot hie:n": "thực hiện",
    "thkrc hi0": "thực hiện",
    "thtrc hi0": "thực hiện",
    "thtyc hi0": "thực hiện",
    "thiyc hilen": "thực hiện",
    "cong viec": "công việc",
    "cong vi?c": "công việc",
    "NOi dung": "Nội dung",
    "Ni dung": "Nội dung",
    "N'Oi dung": "Nội dung",
    "Nt)i dung": "Nội dung",
    "N(ii dung": "Nội dung",
    "Ghi chit": "Ghi chú",
    "Ghi chti": "Ghi chú",
    "Ghi chii": "Ghi chú",

    "Dang 14 mon hoc": "Đăng ký môn học",
    "Dang ky mon hoc": "Đăng ký môn học",
    "Dang IcS,mon hoc": "Đăng ký môn học",
    "mon hoc trkrc tuyen": "môn học trực tuyến",
    "mon hoc true tuyen": "môn học trực tuyến",
    "truc tuyen": "trực tuyến",
    "trkrc tuyen": "trực tuyến",
    "Dong học phi": "Đóng học phí",
    "Dong hoc phi": "Đóng học phí",
    "Thoi gian thi": "Thời gian thi",
    "Thai gian thi": "Thời gian thi",
    "lich thi": "lịch thi",
    "Lich thi": "Lịch thi",
    "C8ng13.8": "Công bố",
    "Cong b6": "Công bố",
    "Cong bo": "Công bố",
    "Giai guy quyet": "Giải quyết",
    "Giai quyet": "Giải quyết",
    "xet tot nghiep": "xét tốt nghiệp",
    "tot nghiep": "tốt nghiệp",
    "mien giam": "miễn giảm",

    "sinh vien": "sinh viên",
    "Sinh vien": "Sinh viên",
    "giang vien": "giảng viên",
    "Giang vien": "Giảng viên",
    "PhOng": "Phòng",
    "Quan ly dao tao": "Quản lý đào tạo",
    "QLDT": "QLĐT",
    "hoc tap": "học tập",
    "thuc tap": "thực tập",
    "tieng Anh": "tiếng Anh",
    "tieng Nhat": "tiếng Nhật",
    "tin chi": "tín chỉ",
    "s6 tín chỉ": "số tín chỉ",
}

def normalize_vietnamese_ocr(text: str) -> str:
  """Sửa lỗi tiếng Việt phổ biến sau OCR/Docling ở mức toàn văn."""
  if not text:
      return ""

  for wrong, right in VIETNAMESE_OCR_REPLACEMENTS.items():
      text = text.replace(wrong, right)

  regex_replacements = [
        (r"\bThai\s+gian\b", "Thời gian"),
        (r"\bThoi\s+gian\b", "Thời gian"),
        (r"\bthoi\s+gian\b", "thời gian"),
        (r"\bHoc\s*k[ySiI0-9]*\b", "Học kỳ"),
        (r"\bhoc\s*k[ySiI0-9]*\b", "học kỳ"),
        (r"\bDang\s+\d+\s+mon\s+hoc\b", "Đăng ký môn học"),
        (r"\bD[aă]ng\s+k[yý]\s+m[oô]n\s+h[oọ]c\b", "Đăng ký môn học"),
        (r"\btr[uư]c\s+tuyen\b", "trực tuyến"),
        (r"\btrkrc\s+tuyen\b", "trực tuyến"),
        (r"\bth[uư]c\s+hien\b", "thực hiện"),
        (r"\bth[oi]t\s+hie:n\b", "thực hiện"),
        (r"\bGhi\s+ch[ií]t\b", "Ghi chú"),
        (r"\bN[O0]i\s+dung\b", "Nội dung"),
        (r"\bNi\s+dung\b", "Nội dung"),
        (r"\bCong\s+b[o6]\b", "Công bố"),
        (r"\bTru[ao]c\b", "Trước"),
        (r"\bTruck\b", "Trước"),
        (r"\bTrirdc\b", "Trước"),
        (r"\bTill&\b", "Trước"),
        (r"\bTint&\b", "Trước"),
        (r"\bma\s+s6\b", "mã số"),
        (r"\bma\s+so\b", "mã số"),
    ]

  for pattern, repl in regex_replacements:
      text = re.sub(pattern, repl, text, flags=re.IGNORECASE)

  return text

def remove_garbage_lines(text: str) -> str:
  """Loại dòng OCR rác trước khi chunk, nhưng vẫn giữ heading và bảng markdown.
"""
  cleaned_lines = []

  for line in text.splitlines():
    raw = line.strip()

    if not raw:
      cleaned_lines.append("")
      continue

    if raw.startswith("#") or raw.startswith("|"):
      cleaned_lines.append(raw)
      continue

    if len(raw) <= 2:
      continue

    weird_ratio = len(re.findall(r"[^\w\sÀ-ỹ\.,:;\-\(\)/]", raw)) / max(len(raw), 1)
    tokens = raw.split()
    short_token_ratio = sum(1 for t in tokens if len(t) <= 2) / max(len(tokens), 1)

    has_garbage_token = re.search(
      r"\b(Suoni|upyn|thtrc|hie:n|MOkh6a|C8ng13)\b",
      raw,
      flags=re.IGNORECASE
    ) is not None

    if weird_ratio > 0.35 or (len(tokens) >= 6 and short_token_ratio > 0.65) or has_garbage_token:
      continue

    cleaned_lines.append(raw)

  return "\n".join(cleaned_lines)

def clean_ocr_text(text: str) -> str:
    """Làm sạch text sau khi Docling/trích xuất trước khi chunking."""
    if not text:
      return ""

    text = remove_markdown_noise(text)
    text = normalize_vietnamese_ocr(text)
    text = remove_garbage_lines(text)
    text = re.sub(r"\s+([,.;:])", r"\1", text)
    text = re.sub(r"([,.;:])(?=\S)", r"\1 ", text)
    return normalize_spaces(text)


def remove_none_values(data: Dict[str, Any]) -> Dict[str, Any]:
    """Loại bỏ key có value rỗng để metadata gọn hơn."""
    clean = {}
    for key, value in data.items():
        if value is None:
            continue
        if isinstance(value, str) and not value.strip():
            continue
        if isinstance(value, list):
            value = [v for v in value if v not in [None, ""]]
            if not value:
                continue
        clean[key] = value
    return clean


def to_metadata_value(value: Any) -> Any:
    """Chuẩn hóa value metadata để phù hợp với Chroma/LangChain.

    Một số vector DB chỉ nhận metadata kiểu str/int/float/bool.
    Vì vậy list/dict sẽ được chuyển thành chuỗi JSON.
    """
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return json.dumps(value, ensure_ascii=False)


def flatten_metadata(metadata: Dict[str, Any]) -> Dict[str, Any]:
    """Chuyển metadata phức tạp thành dạng an toàn cho vector database."""
    return {key: to_metadata_value(value) for key, value in metadata.items()}


def make_chunk(
    *,
    text: str,
    source: Path,
    doc_type: str,
    chunk_type: str,
    chunk_id: int,
    page_start: Optional[int] = None,
    page_end: Optional[int] = None,
    title: Optional[str] = None,
    section: Optional[str] = None,
    article: Optional[str] = None,
    extra: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """Tạo một chunk chuẩn JSONL cho RAG.

    Điểm nâng cấp:
    - Vẫn giữ các field top-level để bạn dễ đọc file JSONL.
    - Thêm trường `metadata` để đưa trực tiếp vào vector database.
    - Metadata gồm thông tin nguồn + thông tin ngữ nghĩa.
    """
    safe_name = clean_filename(source.name)
    clean_text = text.strip()

    # 1) Metadata nguồn: luôn có cho mọi chunk.
    base_metadata = {
        "source_path": str(source),
        "source_file": source.name,
        "document_name": source.name,
        "document_stem": safe_name,
        "document_type": doc_type,
        "chunk_type": chunk_type,
        "chunk_id": chunk_id,
        "page_start": page_start,
        "page_end": page_end,
        "title": title,
        "section": section,
        "article": article,
    }

    # 2) Metadata ngữ nghĩa: truyền từ các hàm chunk chuyên biệt.
    semantic_metadata = extra.copy() if extra else {}

    # 3) Gộp metadata, bỏ giá trị rỗng, chuẩn hóa cho vector DB.
    metadata = remove_none_values({**base_metadata, **semantic_metadata})
    vector_metadata = flatten_metadata(metadata)

    # 4) Item JSONL cuối cùng.
    item = {
        "id": f"{safe_name}_{chunk_type}_{chunk_id}",
        "source": str(source),
        "document_name": source.name,
        "document_type": doc_type,
        "chunk_type": chunk_type,
        "chunk_id": chunk_id,
        "page_start": page_start,
        "page_end": page_end,
        "title": title,
        "section": section,
        "article": article,
        "text": clean_text,
        "metadata": vector_metadata,
    }

    # 5) Giữ thêm các semantic field ở top-level để dễ debug bằng mắt.
    # Khi đưa vào vector DB, bạn nên dùng item["metadata"].
    if extra:
        item.update(remove_none_values(extra))

    return item

# ------------------------------------------------------------
# Fallback chunking helper
# ------------------------------------------------------------
def chunk_text_by_paragraph(
    text: str,
    chunk_size: int = DEFAULT_CHUNK_SIZE,
    overlap: int = DEFAULT_OVERLAP
) -> List[str]:
    """Chunk theo đoạn văn, có overlap nhẹ để giữ ngữ cảnh.

    Hàm này được đặt trong nhóm utility để mọi loại tài liệu đều dùng được:
    - regulation;
    - curriculum;
    - academic_plan;
    - student_handbook;
    - fallback general.
    """
    text = clean_ocr_text(text)
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks = []
    current = ""

    for para in paragraphs:
        if len(current) + len(para) + 2 <= chunk_size:
            current = (current + "\n\n" + para).strip()
        else:
            if current:
                chunks.append(current)

                # Overlap theo ký tự để giữ ngữ cảnh, nhưng không quá lớn.
                tail = current[-overlap:] if overlap > 0 else ""
                current = (tail + "\n\n" + para).strip()
            else:
                start = 0
                while start < len(para):
                    end = start + chunk_size
                    chunks.append(para[start:end].strip())
                    if overlap > 0:
                        start = max(end - overlap, start + 1)
                    else:
                        start = end
                current = ""

    if current:
        chunks.append(current)

    return [c for c in chunks if c.strip()]



# **4. Đọc file PDF bằng Docling**

- Docling để convert PDF sang Markdown có cấu trúc.
- Lưu thêm file `.md` để kiểm tra bảng/heading trước khi đưa vào RAG.

Lưu ý: `page` trong output là `logical page` do Markdown được chia lại theo heading, không phải số trang trong file PDF.

In [ ]:
def count_pdf_pages(pdf_path: Path) -> Optional[int]:
    """Đếm số trang PDF bằng PyMuPDF nếu thư viện khả dụng.

    Thông tin này chỉ dùng để đưa vào report/metadata tổng quan.
    Docling không cần PyMuPDF để trích nội dung chính.
    """
    if fitz is None:
        return None
    try:
        doc = fitz.open(pdf_path)
        return len(doc)
    except Exception:
        return None


def export_docling_to_markdown(pdf_path: Path) -> str:
    """Convert PDF sang Markdown bằng Docling
    Docling sẽ cố gắng giữ lại cấu trúc tài liệu như heading, paragraph,
    bullet list và table. Markdown đầu ra thường phù hợp hơn cho bước chunk.
    """
    result = converter.convert(str(pdf_path))
    markdown = result.document.export_to_markdown()
    return clean_ocr_text(markdown)


def split_markdown_into_logical_pages(markdown: str, max_chars: int = DOC_LOGICAL_PAGE_SIZE) -> List[Dict[str, Any]]:
    """Chia Markdown dài thành các 'trang logic'.

    Vì Docling xuất Markdown theo toàn tài liệu, không phải lúc nào cũng giữ được
    số trang PDF gốc ở dạng đơn giản. Để các hàm chunk cũ vẫn hoạt động,
    ta chia Markdown thành các đoạn logic lớn dựa trên heading.

    Lưu ý:
    - `page` ở đây là logical_page, không nhất thiết là page vật lý trong PDF.
    - Metadata vẫn có `method = docling_markdown` để biết nguồn text đến từ Docling.
    """
    markdown = clean_ocr_text(markdown)
    if not markdown:
        return []

    # Tách theo heading Markdown (#, ##, ###) nhưng vẫn giữ heading trong section.
    sections = re.split(r"(?=^#{1,6}\s+)", markdown, flags=re.MULTILINE)
    sections = [s.strip() for s in sections if s.strip()]

    # Nếu Docling không tạo heading rõ, fallback tách theo đoạn văn.
    if not sections:
        sections = [p.strip() for p in re.split(r"\n\s*\n", markdown) if p.strip()]

    logical_pages = []
    current = ""
    logical_page = 1

    for sec in sections:
        if len(current) + len(sec) + 2 <= max_chars:
            current = (current + "\n\n" + sec).strip()
        else:
            if current:
                logical_pages.append({
                    "page": logical_page,
                    "text": current,
                    "method": "docling_markdown",
                    "is_logical_page": True,
                })
                logical_page += 1
            current = sec

    if current:
        logical_pages.append({
            "page": logical_page,
            "text": current,
            "method": "docling_markdown",
            "is_logical_page": True,
        })

    return logical_pages


def load_pdf_pages(pdf_path: Path, force_ocr: bool = False, zoom: int = 3) -> List[Dict[str, Any]]:
    """Đọc PDF bằng Docling và trả về danh sách logical pages.

    Tham số `force_ocr` và `zoom` được giữ lại để không phá các cell cũ,
    nhưng trong pipeline Docling này chúng không còn là tham số chính.

    Nếu bạn muốn OCR scan PDF trước khi đưa vào Docling, nên dùng workflow ngoài:
    OCRmyPDF → searchable PDF → Docling.
    """
    if force_ocr:
        print("Lưu ý: force_ocr=True không còn dùng trong bản Docling. Docling sẽ tự xử lý PDF theo khả năng của nó.")

    print(f"→ {pdf_path.name}: chuyển PDF sang Markdown bằng Docling")
    markdown = export_docling_to_markdown(pdf_path)

    # Lưu Markdown gốc để kiểm tra cấu trúc bảng/heading.
    safe_name = clean_filename(pdf_path.name)
    md_path = MARKDOWN_DIR / f"{safe_name}_docling.md"
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(markdown)

    pages = split_markdown_into_logical_pages(markdown)

    # Gắn thêm thông tin số trang PDF vật lý nếu đếm được.
    physical_pages = count_pdf_pages(pdf_path)
    for p in pages:
        p["physical_page_count"] = physical_pages
        p["markdown_path"] = str(md_path)

    return pages


def save_pages_text(pdf_path: Path, pages: List[Dict[str, Any]]) -> Path:
    """Lưu text/Markdown đã chia theo logical page để kiểm tra thủ công.

    File này giúp bạn xem Docling đã đọc tài liệu tốt chưa trước khi chunk.
    """
    safe_name = clean_filename(pdf_path.name)
    txt_path = TEXT_DIR / f"{safe_name}_docling_pages.txt"

    parts = []
    for p in pages:
        parts.append(
            f"===== LOGICAL PAGE {p['page']} | METHOD: {p['method']} =====\n"
            f"{p['text']}"
        )

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("\n\n".join(parts))

    return txt_path


## **4.1. Kiểm tra nhanh Markdown do Docling tạo**

In [ ]:
def preview_docling_markdown(pdf_path: Path, max_chars: int = 3000):
    """Convert thử một PDF bằng Docling và in ra phần đầu Markdown.

    Dùng cell này trước khi chạy hàng loạt nếu bạn muốn kiểm tra Docling
    có giữ được bảng/heading tốt không.
    """
    markdown = export_docling_to_markdown(Path(pdf_path))
    print(markdown[:max_chars])
    return markdown

# Ví dụ sử dụng:
# preview_docling_markdown(INPUT_DIR / "ten_file.pdf")

## **5. Nhận diện loại tài liệu / văn bản**

=> Phân loại PDF dựa trên tên file và nội dung để đưa chiến lược chunk phù hợp. Các loại tài liệu được nhận diện:
- tuition: học phí
- regulation: quy chế đào tạo
- curriculum: chương trình đào tạo
- academic_plan: kế hoạch đào tạo /  học vụ
- student_handbook: sổ tay sinh viên
- general: fallback khi chưa nhận diện được loại

Ưu tiên tên file vì bộ dữ liệu có tên file khá rõ để nhận diện tài liệu.

In [ ]:
def full_text_from_pages(pages: List[Dict[str, Any]]) -> str:
    """Ghép text các trang để phục vụ detect loại tài liệu."""
    return "\n\n".join(p.get("text", "") for p in pages)


def detect_document_type(pdf_path: Path, pages: List[Dict[str, Any]]) -> str:
    """Nhận diện loại tài liệu dựa trên tên file và nội dung.
       Sau khi xét tên file, mới xét nội dung để tránh bị nhận diện sai loại tài liệu.
    """
    name = pdf_path.name.lower()
    stem = pdf_path.stem.lower()
    text = full_text_from_pages(pages).lower()
    sample = name + "\n" + text[:15000]

    # ============================================================
    # 1. ƯU TIÊN THEO TÊN FILE
    # ============================================================

    # Sổ tay sinh viên
    if (
        "stsv" in name
        or "sotay" in name
        or "so_tay" in name
        or "sổ_tay" in name
        or "so-tay" in name
        or "student_handbook" in name
    ):
        return "student_handbook"

    # CTĐT / chương trình đào tạo
    if (
        "ctdt" in name
        or "ctđt" in name
        or "chuong_trinh_dao_tao" in name
        or "chương_trình_đào_tạo" in name
        or "curriculum" in name
    ):
        return "curriculum"

    # Quy chế
    if (
        "quyche" in name
        or "quy_che" in name
        or "quy-chế" in name
        or "quy_che_dao_tao" in name
        or "quychedaotao" in name
    ):
        return "regulation"

    # Kế hoạch đào tạo
    if (
        "ke_hoach" in name
        or "kế_hoạch" in name
        or "kh_" in name
        or "kh-" in name
        or "kh đào tạo" in name
        or "kh_đào_tạo" in name
        or "dao_tao_trinh_do" in name
        or "đào_tạo_trình_độ" in name
    ):
        return "academic_plan"

    # Học phí
    if (
        "hoc_phi" in name
        or "học_phí" in name
        or "hocphi" in name
        or "học phí" in name
        or "tuition" in name
    ):
        return "tuition"

    # ============================================================
    # 2. XÉT NỘI DUNG SAU KHI TÊN FILE KHÔNG ĐỦ RÕ
    # ============================================================

    # Sổ tay sinh viên: ưu tiên cao hơn regulation vì sổ tay có thể trích quy chế bên trong.
    if (
        "sổ tay sinh viên" in sample
        or "so tay sinh vien" in sample
        or "thông tin dành cho sinh viên" in sample
        or "thong tin danh cho sinh vien" in sample
        or "công tác sinh viên" in sample
        or "cong tac sinh vien" in sample
        or "nơi sinh viên liên hệ" in sample
        or "noi sinh vien lien he" in sample
    ):
        return "student_handbook"

    # Kế hoạch đào tạo: ưu tiên trước học phí vì kế hoạch có thể chứa mục học phí.
    if (
        "kế hoạch đào tạo" in sample
        or "ke hoach dao tao" in sample
        or "kế hoạch đào tạo năm học" in sample
        or "đăng ký môn học" in sample
        or "dang ky mon hoc" in sample
        or "thời gian thực hiện" in sample
        or "thoi gian thuc hien" in sample
        or "công việc" in sample and "học kỳ" in sample
    ):
        return "academic_plan"

    # Học phí: chỉ nhận nếu nội dung thật sự là bảng/mức học phí.
    if (
        "học phí dự kiến" in sample
        or "mức học phí" in sample
        or "muc hoc phi" in sample
        or "mức học phí bình quân" in sample
        or "học phí bình quân" in sample
    ):
        return "tuition"

    # CTĐT: đặt trước regulation nếu có dấu hiệu chương trình đào tạo rõ.
    if (
        "chương trình đào tạo" in sample
        or "chuong trinh dao tao" in sample
        or "programme learning outcomes" in sample
        or "programme contents" in sample
        or "mã môn học" in sample
        or "ma mon hoc" in sample
        or "course code" in sample
        or "course overview" in sample
        or "mô tả môn học" in sample
    ):
        return "curriculum"

    # Quy chế: đặt sau CTĐT để tránh CTĐT có chữ "Điều" bị bắt nhầm.
    if (
        "quy chế đào tạo" in sample
        or "quy che dao tao" in sample
        or (
            ("quy chế" in sample or "quy che" in sample)
            and len(re.findall(r"\bđiều\s+\d+\b|\bdieu\s+\d+\b", sample)) >= 3
        )
    ):
        return "regulation"

    return "general"

# **6. Chunk fallback theo đoạn văn**

=> Dùng cho tài liệu không có cấu trúc rõ / 1 phần quá dài. Khác với kiểu cắt ký tự thô, hàm ưu tiên cắt theo đoạn, hạn chế cắt ngang câu; đồng thời có overlap nhẹ để giữ ngữ cảnh giữa các chunk.

In [ ]:
def fallback_chunks(pdf_path: Path, pages: List[Dict[str, Any]], doc_type: str) -> List[Dict[str, Any]]:
    """Tạo chunk fallback có metadata page_start/page_end đơn giản."""
    chunks = []
    idx = 0
    for p in pages:
        page_text = p.get("text", "")
        for part in chunk_text_by_paragraph(page_text):
            chunks.append(make_chunk(
                text=part,
                source=pdf_path,
                doc_type=doc_type,
                chunk_type="paragraph",
                chunk_id=idx,
                page_start=p["page"],
                page_end=p["page"],
                title=None,
            ))
            idx += 1
    return chunks


# **7. Chunk cho văn bản quy chế**

=> Dùng để xử lý riêng file quy chế đào tạ.
- Nhận diện tiêu đề Điều
- Tách số Điền và tên Điền
- Chunk tài liệu theo từng Điều
- Nếu 1 Điều quá dài thì sẽ chia nhỏ nhưng vẫn giữ metadata của Điều đó.

In [ ]:
def parse_article_title(article_title: Optional[str]) -> Dict[str, Any]:
    """Tách thông tin Điều từ tiêu đề điều khoản.

    Ví dụ: "Điều 16. Đánh giá và tính điểm môn học"
    -> {"article_number": 16, "article_title": "Đánh giá và tính điểm môn học"}
    """
    if not article_title:
        return {}
    m = re.match(r"Điều\s+(\d+)\s*\.?\s*(.*)", article_title, flags=re.IGNORECASE)
    if not m:
        return {"article_raw": article_title}
    return {
        "article_number": int(m.group(1)),
        "article_title": m.group(2).strip() or None,
        "article_raw": article_title,
    }


def split_regulation_articles(page_text: str) -> List[Tuple[str, str]]:
    """Tách text trang thành các đoạn theo Điều.

    Trả về list (article_title, content).
    """
    text = clean_ocr_text(page_text)
    # Bắt các heading dạng: Điều 16. Đánh giá và tính điểm môn học
    pattern = re.compile(r"(?=(?:^|\n)\s*Điều\s+\d+\s*\.?)", flags=re.IGNORECASE)
    parts = [p.strip() for p in pattern.split(text) if p.strip()]

    results = []
    for part in parts:
        m = re.match(r"Điều\s+(\d+)\s*\.?\s*([^\n]*)", part, flags=re.IGNORECASE)
        if m:
            article_title = f"Điều {m.group(1)}. {m.group(2).strip()}".strip()
        else:
            article_title = None
        results.append((article_title, part))
    return results


def chunk_regulation(pdf_path: Path, pages: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Chunk tài liệu quy chế theo Điều.

    Nếu một Điều dài, tiếp tục chia nhỏ theo đoạn nhưng vẫn giữ metadata article.
    Metadata được thêm:
    - chapter
    - article_number
    - article_title
    - article_raw
    - keywords
    """
    chunks = []
    idx = 0
    current_chapter = None
    current_chapter_number = None

    for p in pages:
        text = p.get("text", "")
        # Cập nhật chương nếu phát hiện heading Chương.
        chapter_match = re.search(r"Chương\s+([IVXLC]+|\d+)\s*\n?([^\n]*)", text, flags=re.IGNORECASE)
        if chapter_match:
            current_chapter_number = chapter_match.group(1)
            current_chapter = f"Chương {chapter_match.group(1)} {chapter_match.group(2).strip()}".strip()

        articles = split_regulation_articles(text)
        if not articles:
            articles = [(None, text)]

        for article_title, content in articles:
            article_meta = parse_article_title(article_title)

            if len(content) <= DEFAULT_CHUNK_SIZE * 1.5:
                parts = [content]
            else:
                parts = chunk_text_by_paragraph(content, chunk_size=DEFAULT_CHUNK_SIZE, overlap=DEFAULT_OVERLAP)

            for j, part in enumerate(parts):
                title = article_title or current_chapter or "Quy chế đào tạo"
                if len(parts) > 1 and article_title:
                    title = f"{article_title} — phần {j+1}"

                chunks.append(make_chunk(
                    text=part,
                    source=pdf_path,
                    doc_type="regulation",
                    chunk_type="article",
                    chunk_id=idx,
                    page_start=p["page"],
                    page_end=p["page"],
                    title=title,
                    section=current_chapter,
                    article=article_title,
                    extra={
                        "chapter": current_chapter,
                        "chapter_number": current_chapter_number,
                        **article_meta,
                        "part_index": j + 1,
                        "num_parts": len(parts),
                        "keywords": extract_keywords(part),
                    }
                ))
                idx += 1

    return chunks


# **8. Chunk cho CTĐT**

=> Dùng để xử lí file CTĐT cần trả lời chính xác: ngành có bao nhiêu tín chỉ, mã ngành, môn học nào bao nhiêu tín chỉ, môn nào tiên quyết, chẩu đầu ra PLO là gì

- Trích thông tin tổng quát của CTĐT
- Nhận diện bảng môn học; PLO/CLO
- Tách phần mô tả môn học nếu file có phần Course overview
- Tạo metadata cho từng môn học như mã môn học, tên môn học, số tín chỉ.

In [ ]:

COURSE_CODE_RE = re.compile(r"\b([A-Z]{2,6}\s?-?\d{3,5}[A-Z]?)\b")
PLO_RE = re.compile(r"\bPLO\s*[\.\-]?\s*(\d+)\b", flags=re.IGNORECASE)
CLO_RE = re.compile(r"\bCLO\s*[\.\-]?\s*(\d+)\b", flags=re.IGNORECASE)


def clean_course_text(text: str) -> str:
    """Làm sạch nhẹ text CTĐT."""
    text = clean_ocr_text(text)
    text = text.replace("https: //", "https://").replace("http: //", "http://")
    text = text.replace("www. ", "www.")
    return normalize_spaces(text)


def extract_program_info(text: str) -> Dict[str, Any]:
    """Trích thông tin tổng quát của chương trình đào tạo bằng regex đơn giản."""
    info = {}
    text = clean_course_text(text)

    patterns = {
        "major_vi": r"Tên ngành đào tạo bằng tiếng Việt.*?:\s*([^\n]+)",
        "major_en": r"Major in English.*?:\s*([^\n]+)",
        "major_code": r"Mã ngành.*?:\s*([0-9]{6,8})",
        "level": r"Trình độ đào tạo.*?:\s*([^\n]+)",
        "mode": r"Hình thức đào tạo.*?:\s*([^\n]+)",
        "total_credits": r"Total credits.*?:\s*([0-9]+)|tổng số tín chỉ\).*?:\s*([0-9]+)",
        "degree": r"Văn bằng tốt nghiệp.*?:\s*([^\n]+)",
        "language": r"Ngôn ngữ đào tạo.*?:\s*([^\n]+)",
    }

    for key, pat in patterns.items():
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            val = next((g for g in m.groups() if g), None)
            if val:
                info[key] = val.strip(" .:-")

    for key, pat in {
        "standard_duration": r"Thời gian đào tạo chuẩn\s*:\s*([^\n]+)",
        "min_duration": r"Thời gian học tập tối thiểu\s*:\s*([^\n]+)",
        "max_duration": r"Thời gian học tập tối đa\s*:\s*([^\n]+)",
    }.items():
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            info[key] = m.group(1).strip()

    if "total_credits" in info:
        try:
            info["total_credits"] = int(re.search(r"\d+", str(info["total_credits"])).group(0))
        except Exception:
            pass

    return info


def extract_course_code(text: str) -> Optional[str]:
    """Lấy mã môn học đầu tiên trong text."""
    m = COURSE_CODE_RE.search(text or "")
    return m.group(1).replace(" ", "").replace("-", "") if m else None


def extract_course_name_vi(text: str) -> Optional[str]:
    """Lấy tên môn học tiếng Việt nếu có."""
    patterns = [
        r"Tên tiếng Việt\s*[:\-]\s*(.+)",
        r"Môn học/Course Name\s*[:\-]\s*(.+)",
        r"Tên môn học\s*[:\-]\s*(.+)",
    ]
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            return m.group(1).split("\n")[0].strip()[:250]
    return None


def extract_course_name_en(text: str) -> Optional[str]:
    """Lấy tên môn học tiếng Anh nếu có."""
    patterns = [
        r"Tên tiếng Anh\s*[:\-]\s*(.+)",
        r"Course Name\s*[:\-]\s*(.+)",
    ]
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            return m.group(1).split("\n")[0].strip()[:250]
    return None


def extract_credits(text: str) -> Optional[int]:
    """Lấy số tín chỉ nếu có."""
    patterns = [
        r"Số tín chỉ/Credits\s*[:\-]?\s*(\d+)",
        r"Số tín chỉ\s*[:\-]?\s*(\d+)",
        r"Credits\s*[:\-]?\s*(\d+)",
    ]
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            try:
                return int(m.group(1))
            except Exception:
                pass
    return None


def split_curriculum_headings(text: str) -> List[Dict[str, Any]]:
    """Tách CTĐT theo heading markdown."""
    text = clean_course_text(text)
    lines = text.splitlines()

    sections = []
    current_title = "Mở đầu"
    current_level = 0
    buf = []
    heading_re = re.compile(r"^(#{1,6})\s+(.+)$")

    for line in lines:
        m = heading_re.match(line.strip())
        if m:
            if buf:
                content = "\n".join(buf).strip()
                if len(content) >= 80:
                    sections.append({"title": current_title, "level": current_level, "text": content})
            current_level = len(m.group(1))
            current_title = m.group(2).strip()
            buf = [line]
        else:
            buf.append(line)

    if buf:
        content = "\n".join(buf).strip()
        if len(content) >= 80:
            sections.append({"title": current_title, "level": current_level, "text": content})

    return sections


def detect_curriculum_section_type(title: str, text: str) -> str:
    """Detect section type trong CTĐT."""
    blob = f"{title}\n{text}".lower()

    if any(k in blob for k in ["mô tả môn học", "course overview", "course description"]):
        return "course_overview"

    if any(k in blob for k in ["chuẩn đầu ra", "programme learning outcomes", "plo"]):
        return "plo_section"

    if any(k in blob for k in ["điều kiện tốt nghiệp", "graduation requirement"]):
        return "graduation_requirement"

    if any(k in blob for k in ["nội dung chương trình", "programme contents", "khung chương trình"]):
        return "course_table_section"

    if any(k in blob for k in ["thông tin tổng quát", "general information"]):
        return "program_info"

    return "general_curriculum"


def split_section_preserving_tables(section_text: str, max_chars: int = 4200) -> List[str]:
    """Chia section dài nhưng giữ bảng markdown không bị cắt ngang."""
    section_text = clean_course_text(section_text)
    if len(section_text) <= max_chars:
        return [section_text]

    lines = section_text.splitlines()
    blocks, buf, table = [], [], []
    in_table = False

    def flush_buf():
        nonlocal buf
        if buf:
            blocks.append("\n".join(buf).strip())
            buf = []

    def flush_table():
        nonlocal table
        if table:
            blocks.append("\n".join(table).strip())
            table = []

    for line in lines:
        if line.strip().startswith("|"):
            if not in_table:
                flush_buf()
                in_table = True
            table.append(line)
        else:
            if in_table:
                flush_table()
                in_table = False
            buf.append(line)

    if in_table:
        flush_table()
    else:
        flush_buf()

    chunks, current = [], ""
    for block in blocks:
        candidate = (current + "\n\n" + block).strip() if current else block
        if len(candidate) <= max_chars:
            current = candidate
        else:
            if current:
                chunks.append(current)
            if len(block) > max_chars and not block.strip().startswith("|"):
                chunks.extend(chunk_text_by_paragraph(block, chunk_size=max_chars, overlap=DEFAULT_OVERLAP))
                current = ""
            else:
                current = block

    if current:
        chunks.append(current)

    return [c for c in chunks if len(c.strip()) >= 120]


def split_course_description_blocks(text: str) -> List[str]:
    """Tách từng mô tả môn học trong phần Course overview."""
    text = clean_course_text(text)

    split_re = r"(?=Môn học/Course Name|Tên môn học|Mã môn học/Course Code)"
    parts = re.split(split_re, text)
    blocks = []

    for p in parts:
        p = p.strip()
        if len(p) < 120:
            continue
        if "Mã môn học" in p or "Course Code" in p or extract_course_code(p):
            blocks.append(p)

    return blocks


def extract_plo_blocks(text: str) -> List[Dict[str, str]]:
    """Tách các dòng PLO."""
    blocks = []
    for line in clean_course_text(text).splitlines():
        line = line.strip()
        m = PLO_RE.search(line)
        if m:
            blocks.append({"plo": f"PLO{m.group(1)}", "text": line})
    return blocks


def extract_clo_blocks(text: str) -> List[Dict[str, str]]:
    """Tách các dòng CLO."""
    blocks = []
    for line in clean_course_text(text).splitlines():
        line = line.strip()
        m = CLO_RE.search(line)
        if m:
            blocks.append({"clo": f"CLO{m.group(1)}", "text": line})
    return blocks


def detect_course_category(context: str) -> Optional[str]:
    """Nhận diện nhóm kiến thức của môn học từ context gần đó."""
    lower = context.lower()
    categories = [
        "kiến thức giáo dục đại cương",
        "lý luận chính trị",
        "kiến thức toán, tin học và khoa học tự nhiên",
        "ngoại ngữ",
        "kiến thức giáo dục chuyên nghiệp",
        "kiến thức cơ sở",
        "kiến thức ngành",
        "kiến thức chuyên ngành",
        "kiến thức bổ trợ",
        "tốt nghiệp",
    ]
    for cat in categories:
        if cat in lower:
            return cat
    return None


def chunk_curriculum(pdf_path: Path, pages: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Chunk CTĐT nâng cấp.

    Hỗ trợ:
    - CTĐT có mô tả môn học/course overview;
    - CTĐT chỉ có bảng môn học;
    - PLO/CLO;
    - điều kiện tốt nghiệp;
    - thông tin tổng quát chương trình.
    """
    full_text = full_text_from_pages(pages)
    full_text = clean_course_text(full_text)
    program_info = extract_program_info(full_text)
    sections = split_curriculum_headings(full_text)

    chunks = []
    idx = 0

    for sec in sections:
        title = sec["title"]
        sec_text = sec["text"]
        sec_type = detect_curriculum_section_type(title, sec_text)

        if sec_type == "course_overview":
            blocks = split_course_description_blocks(sec_text)

            for block in blocks:
                course_code = extract_course_code(block)
                course_name_vi = extract_course_name_vi(block)
                course_name_en = extract_course_name_en(block)
                credits = extract_credits(block)
                parts = chunk_text_by_paragraph(block, chunk_size=2600, overlap=120)

                for part_idx, part in enumerate(parts, start=1):
                    title_text = course_name_vi or course_name_en or course_code or "Mô tả môn học"
                    chunks.append(make_chunk(
                        text=part,
                        source=pdf_path,
                        doc_type="curriculum",
                        chunk_type="course_description",
                        chunk_id=idx,
                        page_start=None,
                        page_end=None,
                        title=title_text,
                        section="Mô tả môn học",
                        extra={
                            **program_info,
                            "course_code": course_code,
                            "course_name_vi": course_name_vi,
                            "course_name_en": course_name_en,
                            "credits": credits,
                            "part_index": part_idx,
                            "retrieval_hint": "mô tả môn học, course overview, course description, tiên quyết",
                        }
                    ))
                    idx += 1
            continue

        if sec_type == "plo_section":
            plo_blocks = extract_plo_blocks(sec_text)
            if plo_blocks:
                for p in plo_blocks:
                    chunks.append(make_chunk(
                        text=p["text"],
                        source=pdf_path,
                        doc_type="curriculum",
                        chunk_type="plo",
                        chunk_id=idx,
                        page_start=None,
                        page_end=None,
                        title=p["plo"],
                        section=title,
                        extra={
                            **program_info,
                            "plo": p["plo"],
                            "retrieval_hint": "chuẩn đầu ra chương trình đào tạo",
                        }
                    ))
                    idx += 1
                continue

        clo_blocks = extract_clo_blocks(sec_text)
        if clo_blocks:
            for c in clo_blocks:
                chunks.append(make_chunk(
                    text=c["text"],
                    source=pdf_path,
                    doc_type="curriculum",
                    chunk_type="clo",
                    chunk_id=idx,
                    page_start=None,
                    page_end=None,
                    title=c["clo"],
                    section=title,
                    extra={
                        **program_info,
                        "clo": c["clo"],
                        "retrieval_hint": "chuẩn đầu ra học phần",
                    }
                ))
                idx += 1
            continue

        if sec_type == "course_table_section":
            table_parts = split_section_preserving_tables(sec_text, max_chars=4200)
            for part_idx, part in enumerate(table_parts, start=1):
                chunks.append(make_chunk(
                    text=part,
                    source=pdf_path,
                    doc_type="curriculum",
                    chunk_type="course_table",
                    chunk_id=idx,
                    page_start=None,
                    page_end=None,
                    title=title,
                    section=title,
                    extra={
                        **program_info,
                        "part_index": part_idx,
                        "is_table": "|" in part,
                        "retrieval_hint": "khung chương trình, bảng môn học, số tín chỉ",
                    }
                ))
                idx += 1
            continue

        if sec_type == "graduation_requirement":
            parts = chunk_text_by_paragraph(sec_text, chunk_size=2600, overlap=120)
            for part_idx, part in enumerate(parts, start=1):
                chunks.append(make_chunk(
                    text=part,
                    source=pdf_path,
                    doc_type="curriculum",
                    chunk_type="graduation_requirement",
                    chunk_id=idx,
                    page_start=None,
                    page_end=None,
                    title=title,
                    section=title,
                    extra={
                        **program_info,
                        "part_index": part_idx,
                        "retrieval_hint": "điều kiện tốt nghiệp",
                    }
                ))
                idx += 1
            continue

        # General sections: mục tiêu, vị trí việc làm, chuẩn đầu vào, thông tin tổng quát...
        chunk_type = "program_info" if sec_type == "program_info" else "curriculum_section"
        parts = chunk_text_by_paragraph(sec_text, chunk_size=2800, overlap=120)
        for part_idx, part in enumerate(parts, start=1):
            chunks.append(make_chunk(
                text=part,
                source=pdf_path,
                doc_type="curriculum",
                chunk_type=chunk_type,
                chunk_id=idx,
                page_start=None,
                page_end=None,
                title=title,
                section=title,
                extra={
                    **program_info,
                    "part_index": part_idx,
                    "retrieval_hint": "chương trình đào tạo",
                }
            ))
            idx += 1

    return chunks


# **9. Chunk cho học phí và kế hoạch học vụ**

=> Dùng để xử lý file học phí / kế hoạch học vụ, cần truy vấn theo:
- Ngành / nhóm ngành -> mức học phí
- Công việc / sự kiện => học kỳ/ đợt => ngày bắt đầu - kết thúc

Với file học phí:
- Giữ bảng học phí thành chunk riêng
- Lưu metadata về chương trình chuẩn/tiên tiến
- Nhận diện các mức tiền học phí (nếu có)

Với file Kế hoạch đào tạo:
- Tách theo từng mục kế hoạch
- Giữ nguyên bảng lớn có HK1/HK2/HK3
- Gắn topic như đăng ký môn học, lịch thi, học phí, xét tốt nghiệp

In [ ]:

DATE_RANGE_RE = re.compile(r"(\d{2}/\d{2}/\d{4})\s*[—\-–]\s*(\d{2}/\d{2}/\d{4})")
MONEY_RE = re.compile(r"\b\d{1,3}(?:[,.]\d{3}){2,}\b")


def money_to_int(raw: str) -> Optional[int]:
    """Chuyển '32,000,000' hoặc '32.000.000' thành 32000000."""
    if not raw:
        return None
    digits = re.sub(r"\D", "", raw)
    return int(digits) if digits else None


def infer_program_type(context: str) -> Optional[str]:
    """Nhận diện chương trình chuẩn/tiên tiến từ context học phí."""
    lower = context.lower()
    if "tiên tiến" in lower or "tien tien" in lower:
        return "Chương trình tiên tiến"
    if "chuẩn" in lower or "chuan" in lower:
        return "Chương trình chuẩn"
    return None


def split_markdown_tables_with_context(text: str) -> List[Dict[str, Any]]:
    """Tách bảng Markdown kèm heading gần nhất phía trước."""
    lines = clean_ocr_text(text).splitlines()
    blocks = []
    current_heading = "Mở đầu"
    i = 0

    while i < len(lines):
        line = lines[i].strip()

        if re.match(r"^#{1,6}\s+", line):
            current_heading = re.sub(r"^#{1,6}\s+", "", line).strip()
            i += 1
            continue

        if line.startswith("|"):
            table_lines = []
            while i < len(lines) and lines[i].strip().startswith("|"):
                table_lines.append(lines[i].rstrip())
                i += 1

            table_text = "\n".join(table_lines).strip()
            if len(table_text) > 50:
                blocks.append({
                    "title": current_heading,
                    "text": f"## {current_heading}\n\n{table_text}",
                    "is_table": True,
                })
            continue

        i += 1

    return blocks


def chunk_tuition(pdf_path: Path, pages: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Chunk tài liệu học phí.

    Cách mới:
    - giữ mỗi bảng chương trình là 1 chunk;
    - metadata có program_type, money values;
    - không chia quá nhỏ để tránh mất nhóm ngành.
    """
    full_text = full_text_from_pages(pages)
    table_blocks = split_markdown_tables_with_context(full_text)
    chunks = []
    idx = 0

    for block in table_blocks:
        text = block["text"]
        program_type = infer_program_type(block["title"] + "\n" + text)
        money_values = MONEY_RE.findall(text)

        chunks.append(make_chunk(
            text=text,
            source=pdf_path,
            doc_type="tuition",
            chunk_type="tuition_table",
            chunk_id=idx,
            page_start=None,
            page_end=None,
            title=block["title"],
            section=block["title"],
            extra={
                "program_type": program_type,
                "is_table": True,
                "tuition_fee_values": [money_to_int(v) for v in money_values],
                "retrieval_hint": "học phí, mức học phí bình quân, ngành, nhóm ngành",
            }
        ))
        idx += 1

    if not chunks:
        for p in pages:
            text = p.get("text", "")
            if not text.strip():
                continue
            chunks.append(make_chunk(
                text=text,
                source=pdf_path,
                doc_type="tuition",
                chunk_type="tuition_section",
                chunk_id=idx,
                page_start=p.get("page"),
                page_end=p.get("page"),
                title="Học phí",
                extra={
                    "program_type": infer_program_type(text),
                    "is_table": "|" in text,
                }
            ))
            idx += 1

    return chunks


ACADEMIC_PLAN_TOPIC_RULES = [
    ("dang_ky_mon_hoc", ["đăng ký môn học", "đăng ký học tập", "đăng ký môn học trực tuyến"]),
    ("lich_thi", ["lịch thi", "thời gian thi", "điều chỉnh lịch thi", "xác nhận lịch thi"]),
    ("hoc_phi", ["đóng học phí", "học phí"]),
    ("xet_tot_nghiep", ["xét tốt nghiệp", "khóa luận", "đồ án tốt nghiệp", "bảo vệ kltn"]),
    ("chuyen_nganh", ["chuyển ngành", "ngành thứ hai", "chuyển vào", "chuyển sinh viên"]),
    ("mien_giam_mon_hoc", ["miễn, giảm môn học", "miễn giảm môn học"]),
    ("canh_bao_hoc_vu", ["cảnh báo học vụ", "cảnh báo thời gian học tập"]),
    ("khao_sat", ["khảo sát", "ý kiến sinh viên"]),
    ("tieng_anh_dau_ra", ["tiếng anh đầu ra", "b1", "b2", "bec", "c1", "tkt"]),
    ("lms", ["lms", "hệ thống quản lý học tập"]),
    ("giang_day", ["giảng dạy", "thời khóa biểu", "giảng viên"]),
]


def infer_event_name(context: str) -> Optional[str]:
    """Suy luận tên sự kiện học vụ từ context có ngày tháng."""
    lower = context.lower()
    mapping = {
        "đăng ký môn học": "Đăng ký môn học",
        "dang ky mon hoc": "Đăng ký môn học",
        "đóng học phí": "Đóng học phí",
        "dong hoc phi": "Đóng học phí",
        "thời gian thi": "Thời gian thi",
        "lich thi": "Lịch thi",
        "lịch thi": "Lịch thi",
        "khóa mã số sinh viên": "Khóa mã số sinh viên",
        "mở khóa mã số sinh viên": "Mở khóa mã số sinh viên",
        "xét tốt nghiệp": "Xét tốt nghiệp",
        "tốt nghiệp": "Tốt nghiệp",
        "chuyển ngành": "Chuyển ngành / học ngành thứ hai",
        "miễn": "Xét miễn, giảm môn học",
        "khao sat": "Khảo sát",
        "khảo sát": "Khảo sát",
        "cảnh báo": "Cảnh báo học vụ",
    }
    for key, value in mapping.items():
        if key in lower:
            return value
    return None


def detect_academic_plan_topic(title: str, text: str) -> str:
    """Gán topic cho mục kế hoạch đào tạo."""
    blob = f"{title}\n{text}".lower()
    for topic, keys in ACADEMIC_PLAN_TOPIC_RULES:
        if any(k in blob for k in keys):
            return topic
    return "general_plan"


def split_markdown_by_level2_sections(text: str) -> List[Dict[str, Any]]:
    """Tách Markdown theo heading cấp ##."""
    lines = clean_ocr_text(text).splitlines()
    sections = []
    current_title = "Mở đầu"
    buf = []

    for line in lines:
        if re.match(r"^##\s+", line.strip()):
            if buf:
                sections.append({"title": current_title, "text": "\n".join(buf).strip()})
            current_title = re.sub(r"^##\s+", "", line.strip()).strip()
            buf = [line]
        else:
            buf.append(line)

    if buf:
        sections.append({"title": current_title, "text": "\n".join(buf).strip()})

    return [s for s in sections if len(s["text"]) >= 120]


def split_section_preserving_tables_for_plan(section_text: str, max_chars: int = 4200) -> List[str]:
    """Chia section kế hoạch dài nhưng giữ bảng HK1/HK2/HK3 không bị cắt."""
    section_text = clean_ocr_text(section_text)
    if len(section_text) <= max_chars:
        return [section_text]

    lines = section_text.splitlines()
    blocks, buf, table = [], [], []
    in_table = False

    def flush_buf():
        nonlocal buf
        if buf:
            blocks.append("\n".join(buf).strip())
            buf = []

    def flush_table():
        nonlocal table
        if table:
            blocks.append("\n".join(table).strip())
            table = []

    for line in lines:
        if line.strip().startswith("|"):
            if not in_table:
                flush_buf()
                in_table = True
            table.append(line)
        else:
            if in_table:
                flush_table()
                in_table = False
            buf.append(line)

    if in_table:
        flush_table()
    else:
        flush_buf()

    chunks, current = [], ""
    for block in blocks:
        candidate = (current + "\n\n" + block).strip() if current else block
        if len(candidate) <= max_chars:
            current = candidate
        else:
            if current:
                chunks.append(current)
            if len(block) > max_chars and not block.strip().startswith("|"):
                chunks.extend(chunk_text_by_paragraph(block, chunk_size=max_chars, overlap=DEFAULT_OVERLAP))
                current = ""
            else:
                current = block

    if current:
        chunks.append(current)

    return [c for c in chunks if len(c.strip()) >= 120]


def extract_semester_mentions(text: str) -> List[str]:
    """Trích xuất học kỳ/đợt có trong chunk."""
    found = []
    lower = text.lower()
    for hk in ["học kỳ 1", "học kỳ 2", "học kỳ 3", "hk1", "hk2", "hk3", "đợt 1", "đợt 2", "đợt 3"]:
        if hk in lower:
            found.append(hk.upper().replace("HỌC KỲ", "HK"))
    return sorted(set(found))


def chunk_academic_plan(pdf_path: Path, pages: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Chunk kế hoạch đào tạo theo từng mục kế hoạch, giữ nguyên ngữ cảnh bảng HK1/HK2/HK3."""
    full_text = full_text_from_pages(pages)
    sections = split_markdown_by_level2_sections(full_text)

    chunks = []
    idx = 0

    for section in sections:
        title = section["title"]
        section_text = section["text"]
        topic = detect_academic_plan_topic(title, section_text)

        max_chars = 4200 if "|" in section_text else 3000
        parts = split_section_preserving_tables_for_plan(section_text, max_chars=max_chars)

        for part_idx, part in enumerate(parts, start=1):
            event_name = infer_event_name(part)
            date_ranges = DATE_RANGE_RE.findall(part)

            chunks.append(make_chunk(
                text=part,
                source=pdf_path,
                doc_type="academic_plan",
                chunk_type="plan_section",
                chunk_id=idx,
                page_start=None,
                page_end=None,
                title=title,
                section=title,
                extra={
                    "topic": topic,
                    "event_name": event_name,
                    "part_index": part_idx,
                    "date_ranges": date_ranges,
                    "semester_mentions": extract_semester_mentions(part),
                    "is_table": "|" in part,
                    "retrieval_hint": "kế hoạch đào tạo, học kỳ, deadline, lịch thi, đăng ký môn học, học phí",
                }
            ))
            idx += 1

    if not chunks:
        return fallback_chunks(pdf_path, pages, doc_type="academic_plan")

    return chunks


# **10. Chunk cho Số tay SV / tài liệu hướng dẫn**

=> Dùng để xử lý file sổ tay sinh viên, câu hỏi thường lien quan đến: liên hệ phòng ban; đăng ký ngoại trú; học bổng, miễn giảm học phí, điểm rèn luyện; khung giờ học, địa điểm học.

- Tách nội dung theo heading
- Tách các mục dài thành tiểu mục nhỏ hơn
- Nhận diện phòng ban như Phòng Quản lý đào tạo, Phòng CTSV, Phòng Khảo thí, Thư viện
- Giữ metadata department, section_number, parent_section.

In [ ]:

HEADING_RE = re.compile(
    r"^(#{1,6}\s+.+|PHẦN\s+[IVXLC]+.*|CHƯƠNG\s+[IVXLC]+.*|\d+(?:\.\d+)*\s+[A-ZÀ-Ỹ0-9].*)$",
    flags=re.MULTILINE
)


DEPARTMENT_RULES = [
    ("phong_quan_ly_dao_tao", ["phòng quản lý đào tạo", "quản lý đào tạo"]),
    ("phong_cong_tac_sinh_vien", ["phòng công tác sinh viên", "công tác sinh viên"]),
    ("phong_tai_chinh_ke_toan", ["phòng tài chính", "tài chính kế toán", "học phí"]),
    ("phong_khao_thi", ["phòng khảo thí", "khảo thí", "điểm thi"]),
    ("thu_vien", ["thư viện"]),
    ("doan_hoi", ["đoàn thanh niên", "hội sinh viên", "đoàn - hội"]),
    ("tram_y_te", ["trạm y tế", "bảo hiểm y tế"]),
    ("trung_tam_httt", ["trung tâm quản lý hệ thống thông tin", "hệ thống thông tin"]),
    ("trung_tam_huong_nghiep", ["hướng nghiệp", "tư vấn việc làm"]),
    ("trung_tam_hoc_lieu", ["trung tâm học liệu", "thư quán"]),
]


def parse_heading_info(heading: Optional[str]) -> Dict[str, Any]:
    """Tách số mục và tên mục từ heading của sổ tay."""
    if not heading:
        return {}

    clean_heading = re.sub(r"^#{1,6}\s+", "", heading.strip())

    m = re.match(r"(\d+(?:\.\d+)*)\s+(.+)", clean_heading)
    if m:
        return {"section_number": m.group(1), "section_title": m.group(2).strip()}

    if clean_heading.upper().startswith("PHẦN"):
        return {"part_title": clean_heading.strip()}

    return {"section_title": clean_heading.strip()}


def detect_department(title: str, text: str) -> Optional[str]:
    """Detect phòng ban trong sổ tay."""
    blob = f"{title}\n{text}".lower()
    for dept, keys in DEPARTMENT_RULES:
        if any(k in blob for k in keys):
            return dept
    return None


def split_handbook_headings(text: str) -> List[Dict[str, Any]]:
    """Tách sổ tay theo heading Markdown hoặc heading số mục."""
    text = clean_ocr_text(text)
    lines = text.splitlines()

    sections = []
    current_title = "Mở đầu"
    current_level = 0
    buf = []

    markdown_heading_re = re.compile(r"^(#{1,6})\s+(.+)$")
    plain_heading_re = re.compile(r"^(PHẦN\s+[IVXLC]+.*|CHƯƠNG\s+[IVXLC]+.*|\d+(?:\.\d+)*\s+[A-ZÀ-Ỹ0-9].*)$")

    for line in lines:
        stripped = line.strip()
        md = markdown_heading_re.match(stripped)
        plain = plain_heading_re.match(stripped)

        if md or plain:
            if buf:
                content = "\n".join(buf).strip()
                if len(content) >= 100:
                    sections.append({
                        "title": current_title,
                        "level": current_level,
                        "text": content,
                    })

            if md:
                current_level = len(md.group(1))
                current_title = md.group(2).strip()
            else:
                current_level = 2
                current_title = plain.group(1).strip()

            buf = [line]
        else:
            buf.append(line)

    if buf:
        content = "\n".join(buf).strip()
        if len(content) >= 100:
            sections.append({
                "title": current_title,
                "level": current_level,
                "text": content,
            })

    return sections


def should_split_handbook_section(title: str, text: str) -> bool:
    """Xác định mục sổ tay cần tách nhỏ."""
    lower = f"{title}\n{text}".lower()
    long_section = len(text) > 3000
    important_multi_topic = any(k in lower for k in [
        "nơi sinh viên liên hệ",
        "câu lạc bộ",
        "khoa và các ngành đào tạo",
        "khung thời gian",
        "hệ thống thông tin",
        "quy chế",
    ])
    return long_section or important_multi_topic


def split_handbook_long_section(title: str, text: str) -> List[Dict[str, Any]]:
    """Tách mục dài trong sổ tay theo heading con hoặc bảng."""
    text = clean_ocr_text(text)

    sub_sections = split_handbook_headings(text)
    if len(sub_sections) > 1:
        return sub_sections

    # Nếu không có heading con nhưng quá dài, chia theo paragraph/table preserving.
    parts = split_section_preserving_tables_for_plan(text, max_chars=3000)
    return [{"title": title, "level": None, "text": p} for p in parts]


def chunk_student_handbook(pdf_path: Path, pages: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Chunk sổ tay sinh viên theo heading/tiểu mục/phòng ban.
    - mục dài như "Nơi sinh viên liên hệ" được tách theo phòng ban;
    - bảng được giữ nguyên;
    - metadata có section_number, department, parent_section.
    """
    full_text = full_text_from_pages(pages)
    sections = split_handbook_headings(full_text)

    chunks = []
    idx = 0
    current_part_title = None

    for sec in sections:
        title = sec["title"]
        content = sec["text"]

        if should_split_handbook_section(title, content):
            units = split_handbook_long_section(title, content)
        else:
            units = [sec]

        for unit_idx, unit in enumerate(units, start=1):
            unit_title = unit.get("title", title)
            heading_meta = parse_heading_info(unit_title)

            if heading_meta.get("part_title"):
                current_part_title = heading_meta["part_title"]

            unit_text = unit["text"]

            if len(unit_text) > 3500:
                parts = chunk_text_by_paragraph(unit_text, chunk_size=3000, overlap=120)
            else:
                parts = [unit_text]

            for part_idx, part in enumerate(parts, start=1):
                department = detect_department(unit_title, part)

                chunks.append(make_chunk(
                    text=part,
                    source=pdf_path,
                    doc_type="student_handbook",
                    chunk_type="handbook_section",
                    chunk_id=idx,
                    page_start=None,
                    page_end=None,
                    title=unit_title,
                    section=title,
                    extra={
                        **heading_meta,
                        "part_title": current_part_title,
                        "parent_section": title,
                        "unit_index": unit_idx,
                        "part_index": part_idx,
                        "department": department,
                        "is_table": "|" in part,
                        "retrieval_hint": "sổ tay sinh viên, phòng ban, quy định, hướng dẫn sinh viên",
                    }
                ))
                idx += 1

    if not chunks:
        return fallback_chunks(pdf_path, pages, doc_type="student_handbook")

    return chunks


# **11. Hàm trích keyword đơn giản**

=> Tạo keyword metadata để hỗ trợ debug và có thể dùng cho hybrid search. Chỉ lấy các cụm quan trọng thường gặp trong tài liệu học vụ (không phải keyword extraction)

Các keyword này không thay thế embedding nhưng dùng để
- debug chunk
- Kiểm tra chunk nói về chủ đề gì
- Hỗ trợ hybird search nếu muốn kết hợp keyword search với vector search.

In [ ]:
IMPORTANT_TERMS = [
    "học phí", "đăng ký môn học", "đăng ký học phần", "đăng ký ngoại trú",
    "miễn giảm", "học bổng", "tốt nghiệp", "khóa luận", "thực tập",
    "cảnh báo học tập", "điểm rèn luyện", "điểm trung bình", "tín chỉ",
    "chương trình đào tạo", "chuẩn đầu ra", "mã ngành", "mã môn học",
    "lịch thi", "điều chỉnh lịch thi", "học kỳ", "cố vấn học tập",
    "Phòng Quản lý đào tạo", "Phòng Công tác Sinh viên", "Phòng Khảo thí",
]


def extract_keywords(text: str, max_keywords: int = 12) -> List[str]:
    """Trích keyword đơn giản từ text."""
    found = []
    lower = text.lower()
    for term in IMPORTANT_TERMS:
        if term.lower() in lower:
            found.append(term)
    # Thêm mã môn học nếu có.
    found.extend(COURSE_CODE_RE.findall(text)[:5])
    # Thêm ngày tháng nếu có.
    found.extend(re.findall(r"\d{2}/\d{2}/\d{4}", text)[:5])
    # Loại trùng nhưng giữ thứ tự.
    unique = []
    for item in found:
        if item and item not in unique:
            unique.append(item)
    return unique[:max_keywords]

# **12. Bộ điều phối chunk theo loại tài liệu**

=> Sau khi nhận diện loại tài liệu,  gọi đúng chiến lược chunk tương ứng. Nếu nhận diện sai, có thể dùng manual_doc_type để xử lý riêng file
- Nhận diện tài liệu đã detect
- Gọi đúng hàm chunk tương ứng
- Loại chunk trùng lặp
- Lưu output JSONL
- Tạo file preview để kiểm tra nhanh chất lượng

In [ ]:
def deduplicate_chunks(chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Loại bỏ chunk trùng lặp gần như y hệt."""
    seen = set()
    output = []

    for item in chunks:
        key = re.sub(r"\s+", " ", item.get("text", "").lower()).strip()[:900]
        if key in seen:
            continue
        seen.add(key)
        output.append(item)

    return output


def filter_short_chunks(chunks: List[Dict[str, Any]], min_chars: int = 40) -> List[Dict[str, Any]]:
    """Lọc chunk quá ngắn nhưng vẫn giữ các chunk semantic quan trọng.

    Giữ lại:
    - PLO
    - CLO
    - article/Điều
    - course_description
    - tuition_table
    """
    important_types = {
        "plo",
        "clo",
        "article",
        "course_description",
        "tuition_table",
    }

    filtered_chunks = []

    for c in chunks:
        text = c.get("text", "").strip()
        chunk_type = c.get("chunk_type", "")

        if chunk_type in important_types:
            filtered_chunks.append(c)
            continue

        if len(text) < min_chars:
            continue

        filtered_chunks.append(c)

    return filtered_chunks


def create_optimized_chunks(
    pdf_path: Path,
    pages: List[Dict[str, Any]],
    manual_doc_type: Optional[str] = None
) -> List[Dict[str, Any]]:
    """Tạo chunk tối ưu theo loại tài liệu.

    Bản tích hợp:
    - tuition: giữ bảng học phí;
    - regulation: theo Điều;
    - curriculum: CTĐT nâng cấp course_description/PLO/CLO/course_table;
    - academic_plan: theo từng mục kế hoạch, giữ bảng HK1/HK2/HK3;
    - student_handbook: theo heading/tiểu mục/phòng ban.
    """
    doc_type = manual_doc_type or detect_document_type(pdf_path, pages)
    print(f"Loại tài liệu nhận diện: {doc_type}")

    if doc_type == "tuition":
        chunks = chunk_tuition(pdf_path, pages)
    elif doc_type == "regulation":
        chunks = chunk_regulation(pdf_path, pages)
    elif doc_type == "curriculum":
        chunks = chunk_curriculum(pdf_path, pages)
    elif doc_type == "academic_plan":
        chunks = chunk_academic_plan(pdf_path, pages)
    elif doc_type == "student_handbook":
        chunks = chunk_student_handbook(pdf_path, pages)
    else:
        chunks = fallback_chunks(pdf_path, pages, doc_type="general")

    # Nếu chunk chuyên biệt tạo quá ít chunk, bổ sung fallback để tránh mất thông tin.
    if len(chunks) < 2:
        chunks = fallback_chunks(pdf_path, pages, doc_type=doc_type)

    before_dedup = len(chunks)
    chunks = deduplicate_chunks(chunks)

    before_filter = len(chunks)
    chunks = filter_short_chunks(chunks, min_chars=40)

    print(f"Số chunk ban đầu: {before_dedup}")
    print(f"Số chunk sau dedup: {before_filter}")
    print(f"Số chunk sau dedup + filter: {len(chunks)}")

    return chunks


def save_chunks_jsonl(pdf_path: Path, chunks: List[Dict[str, Any]]) -> Path:
    """Lưu danh sách chunk ra JSONL."""
    safe_name = clean_filename(pdf_path.name)
    jsonl_path = JSONL_DIR / f"{safe_name}_optimized_chunks.jsonl"

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for item in chunks:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    return jsonl_path


def save_chunk_preview(pdf_path: Path, chunks: List[Dict[str, Any]], n: int = 10) -> Path:
    """Lưu preview markdown để dễ kiểm tra chất lượng chunk."""
    safe_name = clean_filename(pdf_path.name)
    preview_path = REPORT_DIR / f"{safe_name}_chunk_preview.md"

    lines = [f"# Preview chunks — {pdf_path.name}\n"]
    for item in chunks[:n]:
        lines.append(f"## {item['id']}")
        lines.append(f"- type: `{item.get('document_type')}` / `{item.get('chunk_type')}`")
        lines.append(f"- page: {item.get('page_start')} - {item.get('page_end')}")
        lines.append(f"- title: {item.get('title')}")
        lines.append("")
        lines.append("```json")
        lines.append(json.dumps(item.get("metadata", {}), ensure_ascii=False, indent=2))
        lines.append("```")
        lines.append("\n```text")
        lines.append(item.get("text", "")[:1500])
        lines.append("```\n")

    with open(preview_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    return preview_path

# **13. Xử lý 1 file PDF**

=> Chạy toàn bộ pipeline cho 1 file.

PDF - Docling Markdown - nhận diện loại tài liệu - chunk tối ưu - lưu txt + JSONL - lưu file preview - trả về report.

In [ ]:
def process_one_pdf(
    pdf_path: Path,
    force_ocr: bool = False,
    zoom: int = OCR_ZOOM,
    manual_doc_type: Optional[str] = None,
) -> Dict[str, Any]:
    """Xử lý một PDF từ đầu đến cuối."""
    print("\n" + "=" * 80)
    print(f"ĐANG XỬ LÝ: {pdf_path.name}")
    print("=" * 80)

    pages = load_pdf_pages(pdf_path, force_ocr=force_ocr, zoom=zoom)
    txt_path = save_pages_text(pdf_path, pages)
    chunks = create_optimized_chunks(pdf_path, pages, manual_doc_type=manual_doc_type)
    jsonl_path = save_chunks_jsonl(pdf_path, chunks)
    preview_path = save_chunk_preview(pdf_path, chunks, n=12)

    report = {
        "pdf": str(pdf_path),
        "document_name": pdf_path.name,
        "detected_type": manual_doc_type or detect_document_type(pdf_path, pages),
        "num_pages": len(pages),
        "num_chunks": len(chunks),
        "txt_path": str(txt_path),
        "jsonl_path": str(jsonl_path),
        "preview_path": str(preview_path),
        "methods": sorted(set(p.get("method", "unknown") for p in pages)),
        "extraction_engine": "docling",
        "physical_page_count": pages[0].get("physical_page_count") if pages else None,
        "markdown_path": pages[0].get("markdown_path") if pages else None,
    }

    print("\nHOÀN TẤT")
    print("- TXT:", txt_path)
    print("- JSONL:", jsonl_path)
    print("- Preview:", preview_path)
    print("- Số chunk:", len(chunks))
    return report


# **14. Xử lý nhiều file PDF**

=> Chạy pipeline cho các file trong thư mục input.

In [ ]:
def process_all_pdfs(input_dir: Path = INPUT_DIR, force_ocr: bool = False, zoom: int = OCR_ZOOM) -> List[Dict[str, Any]]:
    """Xử lý tất cả PDF trong thư mục input."""
    pdf_files = sorted(list(Path(input_dir).glob("*.pdf")) + list(Path(input_dir).glob("*.PDF")))

    if not pdf_files:
        print(f"Không tìm thấy PDF nào trong {input_dir.resolve()}")
        return []

    reports = []
    for pdf_path in pdf_files:
        try:
            report = process_one_pdf(pdf_path, force_ocr=force_ocr, zoom=zoom)
            reports.append(report)
        except Exception as e:
            print(f"LỖI khi xử lý {pdf_path.name}: {e}")
            reports.append({"pdf": str(pdf_path), "error": str(e)})

    report_path = REPORT_DIR / "processing_report.json"
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(reports, f, ensure_ascii=False, indent=2)

    print("\nĐã lưu report:", report_path)
    return reports

# **15. Chạy xử lý hết các file data raw**

Cell xử lý chính.

In [ ]:
# CHẠY XỬ LÝ HÀNG LOẠT
# Bản Docling: mọi PDF đều được chuyển qua Docling để xuất Markdown.
# force_ocr và zoom được giữ lại để tương thích hàm cũ, nhưng không còn là tham số chính.

reports = process_all_pdfs(
    input_dir=INPUT_DIR,
    force_ocr=False,
    zoom=OCR_ZOOM
)

reports

# **16. Kiểm tra chất lượng chunk**

=> Đọc vài dòng JSONL để xem chunk ổn không. Cần check các điểm sau:
- document_type đúng không
- chunk_type hợp lý không
- text lỗi font / OCR nặng không
- title, page_start, page_end giúp truy vết nguồn được không

In [ ]:
def preview_jsonl(jsonl_path: Path, n: int = 5):
    """In thử n chunk đầu tiên của một file JSONL."""
    jsonl_path = Path(jsonl_path)
    print("File:", jsonl_path)
    print("=" * 80)
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            item = json.loads(line)
            print(f"\n--- CHUNK {i+1} ---")
            print("id:", item.get("id"))
            print("document_type:", item.get("document_type"))
            print("chunk_type:", item.get("chunk_type"))
            print("title:", item.get("title"))
            print("page:", item.get("page_start"), "-", item.get("page_end"))
            print("metadata:")
            print(json.dumps(item.get("metadata", {}), ensure_ascii=False, indent=2)[:1500])
            print("text preview:")
            print(item.get("text", "")[:1000])

# Ví dụ: xem thử file JSONL đầu tiên nếu có
jsonl_files = sorted(JSONL_DIR.glob("*_optimized_chunks.jsonl"))
if jsonl_files:
    preview_jsonl(jsonl_files[0], n=3)
else:
    print("Chưa có file JSONL. Hãy chạy Cell 17 trước.")

## **16.1 Kiểm tra chất lượng metadata**

=> Check xem các chunk đã có metadata đủ tốt chưa trước khi đưa vào RAG. Thống kê các điểm sau:
- Tỷ lệ chunk có trường metadata
- Các key metadata xuất hiện nhiều nhất
- Chunk nào thiếu metadata nguồn cơ bản
- Vài VD metadata theo từng loại chunk

In [ ]:
def audit_metadata(jsonl_path: Path, max_examples: int = 2):
    """Kiểm tra nhanh chất lượng metadata trong file JSONL."""
    jsonl_path = Path(jsonl_path)
    if not jsonl_path.exists():
        print("Không tìm thấy file:", jsonl_path)
        return

    from collections import Counter, defaultdict

    total = 0
    has_metadata = 0
    key_counter = Counter()
    missing_basic = []
    examples = defaultdict(list)

    basic_keys = {"source_file", "document_type", "chunk_type", "page_start"}

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            total += 1
            item = json.loads(line)
            metadata = item.get("metadata", {})

            if metadata:
                has_metadata += 1

            key_counter.update(metadata.keys())

            missing = [k for k in basic_keys if k not in metadata]
            if missing:
                missing_basic.append((item.get("id"), missing))

            ct = item.get("chunk_type", "unknown")
            if len(examples[ct]) < max_examples:
                examples[ct].append({
                    "id": item.get("id"),
                    "metadata": metadata,
                })

    print("TỔNG SỐ CHUNK:", total)
    print("CHUNK CÓ METADATA:", has_metadata, f"({has_metadata / max(total, 1):.1%})")

    print("\nTOP METADATA KEYS:")
    for key, count in key_counter.most_common(30):
        print(f"- {key}: {count}")

    if missing_basic:
        print("\nCẢNH BÁO: Có chunk thiếu metadata nguồn cơ bản")
        for chunk_id, missing in missing_basic[:10]:
            print("-", chunk_id, "thiếu", missing)
    else:
        print("\nOK: Không có chunk thiếu metadata nguồn cơ bản.")

    print("\nVÍ DỤ METADATA THEO chunk_type:")
    for chunk_type, rows in examples.items():
        print("\n###", chunk_type)
        for row in rows:
            print("id:", row["id"])
            print(json.dumps(row["metadata"], ensure_ascii=False, indent=2)[:1200])

# Kiểm tra file đầu tiên nếu đã tạo JSONL
jsonl_files = sorted(JSONL_DIR.glob("*_optimized_chunks.jsonl"))
if jsonl_files:
    audit_metadata(jsonl_files[0])
else:
    print("Chưa có file JSONL. Hãy chạy Cell 17 trước.")


# **17. Gộp file JSONL thành 1 file**

=> Tạo file chung để đưa vào embedding/vetcor database

In [ ]:
combined_jsonl = JSONL_DIR / "all_documents_optimized_chunks.jsonl"

with open(combined_jsonl, "w", encoding="utf-8") as fout:
    for path in sorted(JSONL_DIR.glob("*_optimized_chunks.jsonl")):
        if path.name == combined_jsonl.name:
            continue
        with open(path, "r", encoding="utf-8") as fin:
            for line in fin:
                fout.write(line)

print("Đã tạo file gộp:", combined_jsonl)

# Đếm số dòng/chunk trong file gộp
total = 0
with open(combined_jsonl, "r", encoding="utf-8") as f:
    for _ in f:
        total += 1
print("Tổng số chunk:", total)

# **18. Thống kê số chunk theo loại tài liệu**

=> Xem dataset RAG có cân bằng không và phát hiện tài liệu bị chunk bất thường

In [ ]:
from collections import Counter, defaultdict

stats = Counter()
by_doc = defaultdict(Counter)
metadata_keys = Counter()

if combined_jsonl.exists():
    with open(combined_jsonl, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            stats[(item.get("document_type"), item.get("chunk_type"))] += 1
            by_doc[item.get("document_name")][item.get("chunk_type")] += 1
            metadata_keys.update(item.get("metadata", {}).keys())

    print("THỐNG KÊ THEO document_type/chunk_type")
    for key, count in stats.most_common():
        print(key, ":", count)

    print("\nTHỐNG KÊ THEO TỪNG FILE")
    for doc, counter in by_doc.items():
        print("\n", doc)
        for chunk_type, count in counter.most_common():
            print("  ", chunk_type, ":", count)

    print("\nTOP METADATA KEYS TRONG FILE GỘP")
    for key, count in metadata_keys.most_common(30):
        print("  ", key, ":", count)
else:
    print("Chưa có file gộp. Hãy chạy Cell 19 trước.")

# **19. Nén output -> tải về**

In [ ]:
zip_path = shutil.make_archive("rag_outputs", "zip", OUTPUT_DIR)
print("Đã tạo file zip:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Nếu dùng Jupyter local, hãy mở file zip tại:", zip_path)
    print("Thông báo:", e)

# **Lưu ý:**

Khi đưa vào RAG cần:
1. Đưa text vào embedding
2. Lưu metadata cùng vector
3. Khi thiết kế chatbot; khi đưa ra câu trả lời cần trích dẫn nguồn: tên file, trang, điều khoản, mã môn học, học phí / mốc thời gian